# Modelos — T031 (Decision Tree com Spark MLlib)

Notebook do primeiro modelo da M03: **Decision Tree Regressor** para prever `temperature_C`, sem uso de scikit-learn.

## Objetivo (T031)

- Treinar arvore de decisao adequada ao tipo de problema (**regressao**).
- Registrar hiperparametros usados.
- Avaliar no protocolo definido em T030 (`split` 70/30, `seed=42`).


In [1]:
from pathlib import Path
import os
import shutil
import socket

from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, col, lit
from pyspark.sql.types import DoubleType, FloatType, IntegerType, LongType, ShortType, DecimalType
from pyspark.storagelevel import StorageLevel
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import DecisionTreeRegressor
from pyspark.ml.evaluation import RegressionEvaluator

TARGET_COL = "temperature_C"
SEED = 42
TRAIN_RATIO = 0.7

# Em cluster, evite amostrar/limitar a menos que seja por custo
SAMPLE_FRACTION = 1
MAX_ROWS = None
# Docker Compose pode definir DTR_NOTEBOOK_MAX_ROWS para cortar o dataset antes de
# randomSplit (cada linha e avaliada) + assemble; sem isto o preview em train_vec dispara job pesado.
_dtr_rows = os.environ.get("DTR_NOTEBOOK_MAX_ROWS", "").strip()
if _dtr_rows and MAX_ROWS is None:
    MAX_ROWS = int(_dtr_rows)

# No Docker Compose, o repo fica em /home/jovyan/work e o dataset pode estar em /dataset
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

candidate_parquets = [
    Path("/dataset/Indian_Weather_Dataset.parquet"),
    REPO_ROOT / "data" / "Indian_Weather_Dataset.parquet",
]
PARQUET_PATH = next((p for p in candidate_parquets if p.exists()), None)
if PARQUET_PATH is None:
    raise FileNotFoundError(
        "Parquet nao encontrado. Confirme que existe em data/ ou que ./data foi montado em /dataset no docker-compose."
    )

# Kernel do Jupyter/VS Code muitas vezes nao herda JAVA_HOME do PowerShell — sem isso o PySpark falha com JAVA_GATEWAY_EXITED.
if not os.environ.get("JAVA_HOME"):
    ms = Path(r"C:\\Program Files\\Microsoft")
    if ms.is_dir():
        for jdk in sorted(ms.glob("jdk-*-hotspot"), reverse=True):
            if (jdk / "bin" / "java.exe").is_file():
                os.environ["JAVA_HOME"] = str(jdk)
                break
    if not os.environ.get("JAVA_HOME"):
        java_exe = shutil.which("java")
        if java_exe:
            jp = Path(java_exe).resolve()
            if jp.parent.name.lower() == "bin":
                os.environ["JAVA_HOME"] = str(jp.parent.parent)
if not os.environ.get("JAVA_HOME"):
    raise RuntimeError(
        "Defina JAVA_HOME para um JDK 17+ (ex.: Microsoft OpenJDK). No VS Code: Settings > Python > Env File ou env do kernel."
    )

# SPARK_MASTER vazio ('') quebra o Spark; .env do Docker costuma definir spark://spark-master:7077, que nao resolve no Windows host.
_spark_master_raw = os.environ.get("SPARK_MASTER")
spark_master = (_spark_master_raw or "").strip() or "local[4]"
if not spark_master.lower().startswith("local") and spark_master.startswith("spark://"):
    try:
        hostport = spark_master[len("spark://") :].split("/", 1)[0]
        host = hostport.rsplit(":", 1)[0]
        socket.gethostbyname(host)
    except OSError:
        print(f"[aviso] SPARK_MASTER={spark_master!r} nao resolve aqui; usando local[4].")
        spark_master = "local[4]"

def _reset_spark_if_stale() -> None:
    """Se a JVM morreu (ex.: OOM), o PySpark pode deixar gateway zumbi — getOrCreate() quebra com ConnectionRefused."""
    try:
        inst = getattr(SparkSession, "_instantiatedSession", None)
        if inst is not None:
            sc = getattr(inst, "_sc", None)
            if sc is not None:
                try:
                    sc.stop()
                except Exception:
                    pass
    except Exception:
        pass
    try:
        SparkSession._instantiatedSession = None
        SparkSession._activeSession = None
    except Exception:
        pass
    try:
        from pyspark.sql.context import SQLContext

        SQLContext._instantiatedContext = None
    except Exception:
        pass
    SparkContext._gateway = None
    SparkContext._jvm = None
    SparkContext._active_spark_context = None


_reset_spark_if_stale()
try:
    spark.stop()
except Exception:
    pass
_reset_spark_if_stale()

_is_local_master = str(spark_master).lower().startswith("local")
_driver_mem = os.environ.get("SPARK_DRIVER_MEMORY") or (
    "8g" if _is_local_master else "4g"
)
_shuffle_parts = os.environ.get("SPARK_SQL_SHUFFLE_PARTITIONS", "16")
_default_par = os.environ.get("SPARK_DEFAULT_PARALLELISM", "8")

builder = (
    SparkSession.builder.appName("T031_DecisionTreeRegressor")
    .master(spark_master)
    .config("spark.driver.memory", _driver_mem)
    .config("spark.sql.shuffle.partitions", _shuffle_parts)
    .config("spark.default.parallelism", _default_par)
)
if _is_local_master:
    _max_result = os.environ.get("SPARK_DRIVER_MAX_RESULT_SIZE", "2g")
    builder = builder.config("spark.driver.maxResultSize", _max_result)
if not str(spark_master).lower().startswith("local"):
    driver_host = os.environ.get("SPARK_DRIVER_HOST", "spark-notebook")
    _exec_mem = os.environ.get("SPARK_EXECUTOR_MEMORY", "1g")
    _exec_cores = os.environ.get("SPARK_EXECUTOR_CORES", "1")
    _cores_max = os.environ.get("SPARK_CORES_MAX", "4")
    builder = (
        builder.config("spark.driver.host", driver_host)
        .config("spark.driver.bindAddress", "0.0.0.0")
        .config("spark.executor.memory", _exec_mem)
        .config("spark.executor.cores", _exec_cores)
        .config("spark.cores.max", _cores_max)
        .config("spark.dynamicAllocation.enabled", "false")
        .config("spark.network.timeout", "600s")
        .config("spark.executor.heartbeatInterval", "120s")
    )

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("REPO_ROOT:", REPO_ROOT)
print("PARQUET_PATH:", PARQUET_PATH)
print("SPARK_MASTER:", spark_master)
print("spark.sparkContext.master:", spark.sparkContext.master)
print("spark.version:", spark.version)
print("spark.driver.memory (pedido):", _driver_mem)
if not str(spark_master).lower().startswith("local"):
    print("spark.driver.host:", driver_host)
    print(
        "spark.executor (mem/cores/max):",
        os.environ.get("SPARK_EXECUTOR_MEMORY", "1g"),
        os.environ.get("SPARK_EXECUTOR_CORES", "1"),
        os.environ.get("SPARK_CORES_MAX", "4"),
    )
print("SAMPLE_FRACTION:", SAMPLE_FRACTION)
print("MAX_ROWS:", MAX_ROWS)


REPO_ROOT: /home/jovyan/work
PARQUET_PATH: /dataset/Indian_Weather_Dataset.parquet
SPARK_MASTER: local[4]
spark.sparkContext.master: local[4]
spark.version: 3.2.1
spark.driver.memory (pedido): 3g
SAMPLE_FRACTION: 1
MAX_ROWS: 800000


In [2]:
try:
    df = spark.read.parquet(str(PARQUET_PATH))
except Exception as exc:
    msg = str(exc)
    if "getSubject is not supported" in msg:
        raise RuntimeError(
            "Falha do Spark/Hadoop com Java atual (getSubject). "
            "Use Java 17 para o processo do Jupyter e reinicie o kernel.\n"
            "Exemplo PowerShell (ajuste o caminho):\n"
            "$env:JAVA_HOME='C:\\Program Files\\Java\\jdk-17'\n"
            "$env:Path=\"$env:JAVA_HOME\\bin;\" + $env:Path\n"
            "Depois reabra o Jupyter e rode novamente o notebook."
        ) from exc
    raise

numeric_types = (DoubleType, FloatType, IntegerType, LongType, ShortType, DecimalType)
exclude_cols = {TARGET_COL, "rain_label"}
feature_cols = [
    f.name
    for f in df.schema.fields
    if isinstance(f.dataType, numeric_types) and f.name not in exclude_cols
]

if not feature_cols:
    raise ValueError("Nenhuma feature numerica encontrada para o treino.")

model_df = df.select([TARGET_COL] + feature_cols).na.drop(subset=[TARGET_COL])
model_df = model_df.fillna(0.0, subset=feature_cols)

# Reducao de volume para notebook local (evita queda da JVM/Py4J)
if SAMPLE_FRACTION < 1.0:
    model_df = model_df.sample(withReplacement=False, fraction=SAMPLE_FRACTION, seed=SEED)

if MAX_ROWS is not None and MAX_ROWS > 0:
    model_df = model_df.limit(MAX_ROWS)

# `coalesce()` reduz partições sem shuffle completo (evita pico de memória na fase de metadados da árvore).
model_df = model_df.coalesce(8)
# DISK_ONLY: MEMORY_AND_DISK com milhoes de linhas rebenta o driver em local[4] (3g heap).
model_df = model_df.persist(StorageLevel.DISK_ONLY)

print("N features:", len(feature_cols))
print("Features usadas:", feature_cols)
print("Preview dos dados usados no treino:")
model_df.limit(5).show(truncate=False)


N features: 17
Features usadas: ['lat', 'lon', 'humidity_pct', 'pressure_hPa', 'dew_point_C', 'pressure_trend', 'solar_radiation_Wm2', 'wind_speed_ms', 'cloud_cover_pct', 'hour', 'month', 'wind_direction_deg', 'wind_dir_sin', 'wind_dir_cos', 'cape', 'et0_mm', 'precip_mm']
Preview dos dados usados no treino:
+-------------+-------+------+------------+------------+-----------+--------------+-------------------+-------------+---------------+----+-----+------------------+------------------+-------------------+----+------+---------+
|temperature_C|lat    |lon   |humidity_pct|pressure_hPa|dew_point_C|pressure_trend|solar_radiation_Wm2|wind_speed_ms|cloud_cover_pct|hour|month|wind_direction_deg|wind_dir_sin      |wind_dir_cos       |cape|et0_mm|precip_mm|
+-------------+-------+------+------------+------------+-----------+--------------+-------------------+-------------+---------------+----+-----+------------------+------------------+-------------------+----+------+---------+
|20.6         |1

In [3]:
# randomSplit aplica rand() a todas as linhas de model_df — e pesado com datasets grandes.
train_df, test_df = model_df.randomSplit([TRAIN_RATIO, 1.0 - TRAIN_RATIO], seed=SEED)

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
train_vec = assembler.transform(train_df).select(col(TARGET_COL).alias("label"), "features")
test_vec = assembler.transform(test_df).select(col(TARGET_COL).alias("label"), "features")

# Evita jobs pesados no inicio; apenas smoke-check leve.
print("Preview treino:")
train_vec.limit(3).show(truncate=False)
print("Preview teste:")
test_vec.limit(3).show(truncate=False)


Preview treino:
+-----+--------------------------------------------------------------------------------------------------------------------+
|label|features                                                                                                            |
+-----+--------------------------------------------------------------------------------------------------------------------+
|2.2  |[27.1767,78.0081,100.0,998.7,2.2,0.0,0.0,8.3,100.0,1.0,12.0,92.0,0.9993908270190958,-0.0348994967025009,0.0,0.0,0.0]|
|2.4  |[27.1767,78.0081,99.0,998.4,2.3,-0.3,0.0,5.5,100.0,2.0,12.0,79.0,0.981627183447664,0.1908089953765449,0.0,0.0,0.0]  |
|2.7  |[27.1767,78.0081,92.0,997.2,1.6,0.7,5.0,7.5,0.0,7.0,1.0,343.0,-0.2923717047227371,0.9563047559630352,0.0,0.0,0.0]   |
+-----+--------------------------------------------------------------------------------------------------------------------+

Preview teste:
+-----+--------------------------------------------------------------------------------------

In [4]:
# Hiperparametros (modo local leve)
params = {
    "maxDepth": 8,
    "impurity": "variance",
    "minInstancesPerNode": 100,
    "minInfoGain": 0.0,
    "seed": SEED,
}

dtr = DecisionTreeRegressor(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    maxDepth=params["maxDepth"],
    impurity=params["impurity"],
    minInstancesPerNode=params["minInstancesPerNode"],
    minInfoGain=params["minInfoGain"],
    seed=params["seed"],
)

model = dtr.fit(train_vec)
pred_dt = model.transform(test_vec)
print("Modelo treinado.")


Modelo treinado.


In [5]:
def eval_reg(pred_df):
    mae = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="mae").evaluate(pred_df)
    rmse = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse").evaluate(pred_df)
    r2 = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2").evaluate(pred_df)
    return {"MAE": float(mae), "RMSE": float(rmse), "R2": float(r2)}

# Baseline para comparacao no mesmo protocolo
baseline_value = train_vec.select(avg("label").alias("avg_label")).first()["avg_label"]
pred_base = test_vec.withColumn("prediction", lit(float(baseline_value)))

# Predicao no treino para checar overfitting
pred_dt_train = model.transform(train_vec)

m_base = eval_reg(pred_base)
m_dt_test = eval_reg(pred_dt)
m_dt_train = eval_reg(pred_dt_train)

rows_teste = [
    ("baseline_media_global", m_base["MAE"], m_base["RMSE"], m_base["R2"]),
    ("decision_tree_regressor", m_dt_test["MAE"], m_dt_test["RMSE"], m_dt_test["R2"]),
]

rows_gap = [
    ("MAE", m_dt_train["MAE"], m_dt_test["MAE"], m_dt_test["MAE"] - m_dt_train["MAE"]),
    ("RMSE", m_dt_train["RMSE"], m_dt_test["RMSE"], m_dt_test["RMSE"] - m_dt_train["RMSE"]),
    ("R2", m_dt_train["R2"], m_dt_test["R2"], m_dt_train["R2"] - m_dt_test["R2"]),
]

print("Hiperparametros Decision Tree:")
for k, v in params.items():
    print(f"- {k}: {v}")

print("\nMetricas (teste):")
print(f"{'modelo':<26} {'MAE':>10} {'RMSE':>10} {'R2':>10}")
print("-" * 60)
for modelo, mae, rmse, r2 in rows_teste:
    print(f"{modelo:<26} {mae:>10.4f} {rmse:>10.4f} {r2:>10.4f}")
print("-" * 60)

print("\nDecision Tree: treino vs teste")
print(f"{'split':<10} {'MAE':>10} {'RMSE':>10} {'R2':>10}")
print("-" * 44)
print(f"{'treino':<10} {m_dt_train['MAE']:>10.4f} {m_dt_train['RMSE']:>10.4f} {m_dt_train['R2']:>10.4f}")
print(f"{'teste':<10} {m_dt_test['MAE']:>10.4f} {m_dt_test['RMSE']:>10.4f} {m_dt_test['R2']:>10.4f}")
print("-" * 44)

print("\nGap de generalizacao (teste - treino para erro; treino - teste para R2):")
for nome, tr, te, gap in rows_gap:
    print(f"- {nome}: treino={tr:.4f} | teste={te:.4f} | gap={gap:+.4f}")


Hiperparametros Decision Tree:
- maxDepth: 8
- impurity: variance
- minInstancesPerNode: 100
- minInfoGain: 0.0
- seed: 42

Metricas (teste):
modelo                            MAE       RMSE         R2
------------------------------------------------------------
baseline_media_global          4.8341     6.4013    -0.0000
decision_tree_regressor        0.7761     1.0809     0.9715
------------------------------------------------------------

Decision Tree: treino vs teste
split             MAE       RMSE         R2
--------------------------------------------
treino         0.7791     1.0823     0.9716
teste          0.7761     1.0809     0.9715
--------------------------------------------

Gap de generalizacao (teste - treino para erro; treino - teste para R2):
- MAE: treino=0.7791 | teste=0.7761 | gap=-0.0030
- RMSE: treino=1.0823 | teste=1.0809 | gap=-0.0013
- R2: treino=0.9716 | teste=0.9715 | gap=+0.0001


In [6]:
# Importancias das features (apoio de interpretacao)
importances = model.featureImportances.toArray().tolist()
feat_imp = sorted(zip(feature_cols, importances), key=lambda x: x[1], reverse=True)
print("Top 15 importancias:")
for name, imp in feat_imp[:15]:
    print(f"- {name}: {imp:.6f}")


Top 15 importancias:
- dew_point_C: 0.351142
- et0_mm: 0.332209
- humidity_pct: 0.294348
- pressure_hPa: 0.008707
- month: 0.007093
- solar_radiation_Wm2: 0.004244
- hour: 0.002125
- lat: 0.000126
- wind_direction_deg: 0.000006
- lon: 0.000000
- pressure_trend: 0.000000
- wind_speed_ms: 0.000000
- cloud_cover_pct: 0.000000
- wind_dir_sin: 0.000000
- wind_dir_cos: 0.000000


In [7]:
# Opcional: salvar modelo treinado para demo/reuso
# Em Windows, o writer do Spark pode falhar sem configuracao Hadoop/winutils.
MODEL_DIR = REPO_ROOT / "models" / "decision_tree_regressor_t031"
MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)

try:
    model.write().overwrite().save(str(MODEL_DIR))
    print("Modelo salvo em:", MODEL_DIR)
except Exception as exc:
    print("Aviso: nao foi possivel salvar o modelo Spark neste ambiente local.")
    print("Motivo:", str(exc)[:300], "...")
    print("Treino e metricas seguem validos; apenas o artefato Spark nao foi persistido.")


Modelo salvo em: /home/jovyan/work/models/decision_tree_regressor_t031


In [8]:
# Encerrar sessao Spark ao fim da execucao
spark.stop()
print("Spark finalizado.")


Spark finalizado.
